In [1]:
!pip install sentinelsat asf_search

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.8/48.8 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.3/115.3 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 315.5/315.5 kB 13.5 MB/s eta 0:00:00


In [2]:
import os
from sentinelsat import SentinelAPI, read_geojson, geojson_to_wkt
import asf_search as asf
from shapely.geometry import box
import requests
import json
import shutil

In [7]:
CDSE_EMAIL = 'rdas52571@gmail.com'
CDSE_PASS = 'R@hul8257013'
ASF_USER = 'rahulds001'
ASF_PASS = 'R@hul8257013'
OUTPUT_DIR = "/content/drive/MyDrive/Galaxeye Task"

In [4]:
def get_copernicus_token(email, password):

    auth_url = "https://identity.dataspace.copernicus.eu/auth/realms/CDSE/protocol/openid-connect/token"

    payload = {
        "client_id": "cdse-public",
        "username": email,
        "password": password,
        "grant_type": "password",
    }

    response = requests.post(auth_url, data=payload)

    if response.status_code == 200:
        return response.json()["access_token"]
    else:
        print(f"Auth failed! Status: {response.status_code}")
        print(response.text)
        return None

In [5]:
ROI_WKT = "POLYGON((92.35 26.35, 92.55 26.35, 92.55 26.50, 92.35 26.50, 92.35 26.35))"

In [12]:
def download_sentinel2_optical():
    print("\n--- Starting Sentinel-2 (Optical) Search ---")

    # 1. Get Authentication
    token = get_copernicus_token(CDSE_EMAIL, CDSE_PASS)
    if not token:
        return

    # 2. Build the Search Query
    # We are looking for a clear image from March 2024 (Dry Season Baseline)
    # The API needs the geometry specifically formatted with SRID=4326
    odata_url = "https://catalogue.dataspace.copernicus.eu/odata/v1/Products"

    query_filters = (
        "Collection/Name eq 'SENTINEL-2' and "
        "ContentDate/Start ge 2024-03-01T00:00:00.000Z and "
        "ContentDate/Start le 2024-03-30T00:00:00.000Z and "
        "Attributes/OData.CSC.DoubleAttribute/any(att:att/Name eq 'cloudCover' and att/OData.CSC.DoubleAttribute/Value le 10.00) and "
        f"OData.CSC.Intersects(area=geography'SRID=4326;{ROI_WKT}')"
    )

    params = {
        "$filter": query_filters,
        "$orderby": "ContentDate/Start asc", # Get the earliest good image
        "$top": 1 # We only need one good scene
    }

    print("Searching catalog...")
    response = requests.get(odata_url, params=params)
    results = response.json()

    if not results['value']:
        print("No images found! Check your dates or cloud cover limits.")
        return

    # 3. Handle the Download
    image = results['value'][0]
    filename = f"{image['Name']}.zip"
    save_path = os.path.join(OUTPUT_DIR, "sentinel2", filename)
    download_link = f"https://catalogue.dataspace.copernicus.eu/odata/v1/Products({image['Id']})/$value"

    print(f"Found Image: {image['Name']}")

    if os.path.exists(save_path):
        print("File already exists. Skipping download.")
        return

    print("Downloading... (This might take a minute)")

    # IMPORTANT: The server redirects us to a storage location (Zipper).
    # Standard requests will drop our Auth headers on redirect, causing a 401 error.
    # We have to handle this manually.
    headers = {"Authorization": f"Bearer {token}"}

    # Step A: Hit the URL but don't follow the jump yet
    initial_req = requests.get(download_link, headers=headers, stream=True, allow_redirects=False)

    # Step B: Grab the new URL and go there WITH our token
    if initial_req.status_code in [301, 302, 303, 307]:
        storage_url = initial_req.headers['Location']

        with requests.get(storage_url, headers=headers, stream=True) as r:
            with open(save_path, 'wb') as f:
                shutil.copyfileobj(r.raw, f)
        print("Download finished successfully!")

    elif initial_req.status_code == 200:
        # Sometimes it works directly
        with open(save_path, 'wb') as f:
            shutil.copyfileobj(initial_req.raw, f)
        print("Download finished successfully!")
    else:
        print(f"Something went wrong. Status Code: {initial_req.status_code}")

In [16]:
def download_sentinel1_sar():
    print("\n--- Starting Sentinel-1 (SAR) Search ---")
    print("Querying Alaska Satellite Facility...")

    # FIX: Widened date range to full month and removed 'DESCENDING' restriction
    # This guarantees we find a pass regardless of orbit schedule
    results = asf.search(
        platform=asf.PLATFORM.SENTINEL1,
        intersectsWith=ROI_WKT,
        start='2024-07-01',
        end='2024-07-30',
        processingLevel=asf.PRODUCT_TYPE.GRD_HD,
        beamMode=asf.BEAMMODE.IW,
        maxResults=1
    )

    print(f"Found {len(results)} scenes.")

    if len(results) == 0:
        print("Still no results? Try expanding the ROI or picking a different month.")
        return

    # Create the session
    try:
        session = asf.ASFSession().auth_with_creds(ASF_USER, ASF_PASS)
    except Exception as e:
        print(f"ASF Auth Failed: {e}")
        return

    # Download
    print(f"Downloading {results[0].properties['sceneName']}...")
    results.download(
        path=os.path.join(OUTPUT_DIR, "sentinel1"),
        session=session
    )
    print("SAR Download complete.")



In [18]:
if __name__ == "__main__":
    # Make sure folders exist
    os.makedirs(os.path.join(OUTPUT_DIR, "sentinel2"), exist_ok=True)
    os.makedirs(os.path.join(OUTPUT_DIR, "sentinel1"), exist_ok=True)

    # Run the jobs
    download_sentinel2_optical()
    download_sentinel1_sar()


--- Starting Sentinel-2 (Optical) Search ---
Searching catalog...
Found Image: S2B_MSIL2A_20240301T042709_N0510_R133_T46RDQ_20240301T072203.SAFE
Downloading... (This might take a minute)
Download finished successfully!

--- Starting Sentinel-1 (SAR) Search ---
Querying Alaska Satellite Facility...
Found 1 scenes.
SAR Download complete.
